# **Experiment Notebook**



In [1]:
# Do not modify this code
!pip install -q utstd

from utstd.ipyrenders import *


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Do not modify this code
import warnings
warnings.simplefilter(action='ignore')

## 0. Import Packages

In [ ]:
# <Student to fill this section>
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
from collections import Counter
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

---
## A. Project Description


In [ ]:
# <Student to fill this section>
student_name = "Prathamesh Nemade"
student_id = "25672914"
group_id = "23"

In [ ]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

In [ ]:
# Do not modify this code
print_tile(size="h1", key='group_id', value=group_id)

---
## B. Experiment Hypothesis

In [ ]:
# <Student to fill this section>
experiment_hypothesis ="""Using SMOTE to rebalance the training set and training XGBoost on the selected important features will outperform the Random Forest and earlier baselines on F1. XGBoost’s ability to learn non-linear interactions should improve recall for the drafted class while keeping precision reasonable.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='experiment_hypothesis', value=experiment_hypothesis)

### C.1   Load Datasets


In [ ]:
# Load data
# -------------------------------
train_df=pd.read_csv("../data/raw/train.csv")
test_df=pd.read_csv("../data/raw/test.csv")

### C.2  Load Packages


In [ ]:
pip install -i https://test.pypi.org/simple/ advml-at1-25672914

In [ ]:
# Import your utility functions
from advml_at1_25672914.data.data_utils import dataset_info, preprocess_data, transform_features,split_data
from advml_at1_25672914.visualization.visualize import (
    plot_confusion_matrix,
    plot_roc_auc,
    plot_precision_recall_curve,
    plot_class_distribution,
    plot_feature_importance
)

In [ ]:
# <Student to fill this section>
Load_Packages_explanations = """
This cell imports several essential functions from custom modules (`advml_at1_25672914.data.data_utils` and `advml_at1_25672914.visualization.visualize`) for preparing and understanding the data.

From `advml_at1_25672914.data.data_utils`, the following functions are imported:

-   `dataset_info`: This function is likely used to provide a summary of the dataset, including its shape, data types, and potentially missing value counts. This helps in the initial understanding of the data structure and identifying potential issues.
-   `preprocess_data`: This function is crucial for handling raw data and making it suitable for modeling. It likely includes steps such as handling missing values (e.g., imputation or removal), and potentially other cleaning operations.
-   `split_data`: This function is used to divide the dataset into training and validation sets. For this imbalanced classification task, it's important that this function performs a *stratified* split, ensuring that the proportion of the target variable (drafted players) is maintained in both sets. This helps in building and evaluating a model that performs well on the minority class.
-   `transform_features`: This function is responsible for applying transformations to the features. This could include techniques like one-hot encoding for categorical variables and scaling numerical features (standardization or normalization). One-hot encoding is necessary because most machine learning algorithms cannot directly process categorical data. Scaling is important for algorithms sensitive to the magnitude of features, ensuring that no single feature dominates the model due to its scale.

From `advml_at1_25672914.visualization.visualize`, the following plotting functions are imported:

-   `plot_confusion_matrix`: This function visualizes the performance of a classification model by showing the counts of true positives, true negatives, false positives, and false negatives. It's particularly useful for understanding model performance on imbalanced datasets.
-   `plot_roc_auc`: This function plots the Receiver Operating Characteristic (ROC) curve and calculates the Area Under the Curve (AUC). The ROC curve illustrates the trade-off between the true positive rate and the false positive rate at various threshold settings. AUC provides a single metric summarizing the model's ability to discriminate between positive and negative classes.
-   `plot_precision_recall_curve`: This function plots the precision-recall curve, which is especially informative for imbalanced datasets. It shows the trade-off between precision (the proportion of predicted positives that are truly positive) and recall (the proportion of actual positives that are correctly identified) at different thresholds.
-   `plot_class_distribution`: This function visualizes the distribution of the target variable. In this project, it's used to show the severe imbalance between drafted and non-drafted players, highlighting the need for appropriate evaluation metrics and potentially techniques to address the imbalance.

In summary, these imported packages and functions provide the necessary tools for data loading, cleaning, transformation, splitting, and visualization, which are fundamental steps in building and evaluating a machine learning model for this imbalanced classification problem.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='Load_Packages_explanations', value=Load_Packages_explanations)

### C.3 Explore Target variable

In [ ]:
# -------------------------------
# Dataset overview
# -------------------------------
dataset_info(train_df)
plot_class_distribution(train_df['drafted'], title="Target Class Distribution")

In [ ]:
# <Student to fill this section>
target_distribution_explanations = """The analysis of the `'drafted'` target variable reveals a significant class imbalance, with the vast majority of players not being drafted—often upwards of 95% or more depending on the dataset. This imbalance poses a major challenge for predictive modeling, as models tend to favor the majority class, leading to poor recall for the minority class (drafted players) and misleadingly high accuracy. In such cases, accuracy becomes an unreliable metric, as a model predicting all players as "not drafted" could still achieve high scores while failing to identify any true draft picks. Additional limitations may include limited sample size for the drafted class, potential label noise, or temporal drift if draft criteria change over time. From a business perspective, this skewed distribution reflects the real-world challenge of scouting—only a small fraction of players are selected, so the model must be highly sensitive and precise in identifying those few with draft potential. Addressing this imbalance through techniques like resampling, class weighting, or specialized metrics (e.g., precision, recall, F1-score) is essential for building a useful and fair predictive system.

"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='target_distribution_explanations', value=target_distribution_explanations)

### D.   Data Preparation


In [ ]:
selected_features = [
    # Basic play metrics
    "GP",
    "dunksmade",
    "dunksmiss_dunksmade",
    "twoPM",
    "rimmade",
    "midmade",
    "pts",
    "FTA",
    "FTM",
    "Rec_Rank",
    "stops",
    "dreb",
    "blk",
    "treb",

    # Advanced stats
    "porpag",
    "dporpag",
    "adjoe",
    "bpm",
    "obpm",
    "gbpm",
    "Min_per",
    "TS_per",
    "eFG",
    "TP_per",
    "AST_per",

    # Categoricals
    "team",
    "conf"]

In [ ]:
# -------------------------------
# Separate features and target
# -------------------------------
y = train_df['drafted']
X = train_df[selected_features].copy()
test = test_df[selected_features]

In [ ]:
# <Student to fill this section>
feature_selection_explanations = """We selected features that best capture a player’s overall contribution, efficiency, and context while avoiding redundancy. Core performance indicators such as games played and minutes reflect reliability and exposure, while scoring measures (points, shooting accuracy, free throws, and shot profile) highlight both volume and efficiency. Rebounding, blocks, and defensive metrics provide insight into size and defensive impact, and playmaking features like assist percentage and advanced efficiency ratings capture a player’s ability to create for others and contribute beyond scoring. Contextual attributes such as team, conference, and height were also included, as they influence how performance is evaluated by scouts. We excluded identifiers, redundant raw counts, and low-signal statistics since they do not add predictive value. This selection balances volume with efficiency and ensures coverage of scoring, defense, playmaking, and player context, aligning with the factors most relevant in draft decisions.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## E. Data Preparation for Modeling

### E.1 Split Datasets

In [ ]:
# -------------------------------
# Split train into train/validation
# -------------------------------
X_train, X_val, y_train, y_val = split_data(X, y, stratify=True)

In [ ]:
# <Student to fill this section>
data_splitting_explanations = """For this imbalanced classification task, the most effective approach is a **Stratified Train-Validation Split** of the `train.csv` data. Stratification ensures that both the training and validation sets maintain the same proportion of the minority class (`drafted`) as the original dataset, which is critical for learning meaningful patterns and avoiding misleading validation metrics. This setup allows you to tune models and select algorithms without contaminating your final evaluation on the untouched `test.csv`. By preserving class distribution and reserving a validation set, you reduce the risk of overfitting and ensure that your model’s performance generalizes well. An 80/20 split is a common starting point, but can be adjusted based on dataset size and model complexity.

"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_splitting_explanations', value=data_splitting_explanations)

### F.   Data Transformation


### F.1   Data Transformation


In [ ]:
# -------------------------------
# Preprocess after splitting
# -------------------------------
X_train_clean = preprocess_data(X_train)
X_val_clean   = preprocess_data(X_val)
test_clean    = preprocess_data(test)

In [ ]:
# <Student to fill this section>
data_cleaning_1_explanations = """
Handling missing values is a vital preprocessing step because most machine learning algorithms cannot natively process incomplete data. If left unaddressed, missing values can cause errors during training, introduce bias—especially when the missingness is systematic—and lead to the loss of valuable information if rows or columns are dropped indiscriminately. Moreover, they can distort statistical calculations and hinder model performance, resulting in poor generalization, unreliable predictions, and misleading insights into feature importance. Proper imputation or handling strategies are essential to ensure model compatibility, preserve data integrity, and maintain predictive reliability.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### F.2   Data Transformation


In [ ]:
# -------------------------------
# Apply transformations (scaling, one-hot)
# -------------------------------
X_train_transformed = transform_features(X_train_clean)
X_val_transformed   = transform_features(X_val_clean)
test_transformed    = transform_features(test_clean)

In [ ]:
# <Student to fill this section>
data_cleaning_2_explanations = """
Encoding categorical features is essential because most machine learning algorithms require numerical input and cannot directly interpret text-based categories. Without proper encoding, models may misinterpret categorical labels as having ordinal relationships, leading to flawed predictions. Techniques like one-hot encoding, frequency encoding, or target encoding help translate categories into meaningful numerical formats while managing issues like high cardinality and dimensionality. Failing to encode correctly can result in models that are untrainable, biased, or computationally inefficient, ultimately degrading performance and interpretability. Thoughtful encoding ensures compatibility, preserves semantic meaning, and supports robust model development.
Feature scaling is a crucial preprocessing step that standardizes or normalizes the range of input features, ensuring that all variables contribute proportionally during model training. This is especially important for algorithms sensitive to feature magnitude—such as K-Nearest Neighbors, SVMs, and models using gradient descent—where unscaled features can distort distance calculations or dominate optimization. Proper scaling promotes equal feature influence, accelerates convergence, and improves overall model performance. Without it, models may struggle to learn effectively, converge slowly, or yield biased results driven by numerically larger features rather than truly informative ones.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_1_explanations)

### F.3  Data Transformation


In [ ]:
# Handle class imbalance using SMOTE
# -------------------------------
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_transformed, y_train)


In [ ]:
# <Student to fill this section>
data_transformation_3_explanations = """
Applying SMOTE helps address the imbalance between drafted and non-drafted players in the training data. This leads to:
Improved identification of drafted players: The model becomes better at recognizing the characteristics of players who are likely to be drafted.
Reduced bias: The model is less likely to simply predict "not drafted" for everyone.
More useful predictions: The predictions are more valuable for scouting as they are better at highlighting potential draft picks."""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_3_explanations', value=data_transformation_3_explanations)

---
## G. Selection of Performance Metrics

> Provide some explanations on why you believe the performance metrics you chose is appropriate


In [ ]:
from sklearn.metrics import f1_score

In [ ]:
# <Student to fill this section>
performance_metrics_explanations = """ the F1-score is a crucial performance metric because it effectively handles the significant class imbalance between drafted and non-drafted players. Unlike accuracy, which can be misleading when one class is much larger than the other, the F1-score provides a balanced measure by considering both precision and recall for the minority class (drafted players).

Precision is important because it tells us how many of the players the model predicts will be drafted are actually drafted (minimizing wasted scouting effort on false positives).
Recall is vital because it measures how many of the truly drafted players the model correctly identifies (minimizing missed opportunities on false negatives).
The F1-score combines these two aspects, giving us a single metric that reflects the model's ability to both accurately identify potential draftees and avoid overlooking them, which directly aligns with the business objectives of efficient and effective talent scouting."""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='performance_metrics_explanations', value=performance_metrics_explanations)

## H. Train Machine Learning Model

### H.1 Import Algorithm

> Provide some explanations on why you believe this algorithm is a good fit


In [ ]:
from xgboost import XGBClassifier

In [ ]:
# <Student to fill this section>
algorithm_selection_explanations = """XGBoost (Extreme Gradient Boosting) was chosen for this classification task due to its powerful performance on structured data and its ability to handle imbalanced datasets effectively. As a gradient boosting framework, it builds upon weak learners to create a strong predictive model, capturing complex patterns and interactions within the player statistics. Key advantages of XGBoost for this problem include its built-in regularization to prevent overfitting, its handling of missing values, and its flexibility in optimizing various evaluation metrics relevant to imbalanced classification, such as AUC and F1-score. Its speed and scalability also make it suitable for potentially larger datasets.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='algorithm_selection_explanations', value=algorithm_selection_explanations)

### H.2  Set Hyperparameters

> Provide some explanations on why you believe this algorithm is a good fit


In [ ]:
# Train XGBoost classifier
# -------------------------------
# Compute scale_pos_weight for imbalanced dataset
ratio = Counter(y_train_res)[0] / Counter(y_train_res)[1]

xgb_model = XGBClassifier(
    scale_pos_weight=ratio,
    max_depth=6,
    n_estimators=200,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

In [ ]:
import joblib
# Save the model
joblib.dump(xgb_model, "experiment3_xgb_model.pkl")

In [ ]:
# <Student to fill this section>
hyperparameters_selection_explanations = """For this XGBoost model, the hyperparameters were chosen to balance model complexity and prevent overfitting, especially given the class imbalance. `scale_pos_weight` is set to the ratio of negative to positive classes to directly address the imbalance by giving more importance to the minority class (drafted players). `max_depth=6` limits the depth of individual trees to control complexity, while `n_estimators=200` provides enough trees for the ensemble to learn complex patterns. `learning_rate=0.1` controls the step size at each iteration, preventing the model from converging too quickly. `eval_metric='logloss'` is used for evaluation during training, and `random_state=42` ensures reproducibility. The `use_label_encoder=False` parameter is included to suppress a deprecation warning."""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='hyperparameters_selection_explanations', value=hyperparameters_selection_explanations)

### H.3 Fit Model

In [ ]:
xgb_model.fit(X_train_res, y_train_res)

### H.4 Model Technical Performance

> Provide some explanations on model performance


In [ ]:
# Align validation and test set columns with training set
X_val_transformed = X_val_transformed.reindex(columns=X_train_res.columns, fill_value=0)
test_transformed = test_transformed.reindex(columns=X_train_res.columns, fill_value=0)


In [ ]:
# Evaluate on training and validation
y_train_pred = xgb_model.predict(X_train_res)
y_val_pred   = xgb_model.predict(X_val_transformed)

from sklearn.metrics import accuracy_score, f1_score
print("Training Accuracy:", accuracy_score(y_train_res, y_train_pred))
print("Training F1 Score:", f1_score(y_train_res, y_train_pred))
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("Validation F1 Score:", f1_score(y_val, y_val_pred,average='weighted'))


In [ ]:
# Evaluate on validation set
y_val_pred = xgb_model.predict(X_val_transformed)
y_val_proba = xgb_model.predict_proba(X_val_transformed)[:, 1]
f1_val = f1_score(y_val, y_val_pred,average='weighted')
print(f"Validation F1 Score: {f1_val:.4f}")

In [ ]:
# Visualizations
plot_confusion_matrix(y_val, y_val_pred, labels=[0, 1])
plot_roc_auc(y_val, y_val_proba)
plot_precision_recall_curve(y_val, y_val_proba)
plot_feature_importance(xgb_model.feature_importances_, X_train_transformed.columns, top_n=15)


In [ ]:
# Make predictions on test set
test_preds = xgb_model.predict(test_transformed)
test_proba = xgb_model.predict_proba(test_transformed)[:, 1]  # probabilities for class 1

# Combine predictions with player IDs (or whatever identifier you have)
submission = pd.DataFrame({
    "player_id": test_df["player_id"],  # make sure test_df has this column
    "probability": test_proba
})

# Show first few rows
print(submission.head())

# Optionally save to CSV
submission.to_csv("xgb_test_predictions.csv", index=False)
print("Test predictions saved to xgb_test_predictions.csv")


In [ ]:
# <Student to fill this section>
model_performance_explanations = """The XGBoost model, trained on SMOTE-resampled data, shows strong performance with a validation F1 score of 0.9948 and an AUC of 0.99. The confusion matrix indicates minimal false positives and negatives, suggesting the model is effective at identifying both drafted and non-drafted players. The high F1 score is particularly important given the class imbalance, as it provides a balanced measure of precision and recall. Reindexing was necessary to ensure that the columns in the validation and test sets matched the columns in the training set after one-hot encoding and SMOTE, preventing errors during prediction. The feature importance plot highlights key factors influencing predictions, such as Rec_Rank, dreb, and dunksmade, which align with expected indicators of draft potential."""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='model_performance_explanations', value=model_performance_explanations)

### H.5  Business Impact from Current Model Performance

> Provide some analysis on the model impacts from the business point of view


In [ ]:
# <Student to fill this section>
business_impacts_explanations = """The model significantly improves talent scouting by efficiently identifying high-potential players, reducing the risk of missing key talent, and enabling data-driven recruitment decisions. This leads to better resource allocation, enhanced competitiveness, and improved overall recruitment outcomes. Ongoing refinement is key to maintaining its value."""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='business_impacts_explanations', value=business_impacts_explanations)

## I. Project Outcomes

In [ ]:
# <Student to fill this section
experiment_outcome = "" #'Hypothesis Confirmed'

In [ ]:
# <Student to fill this section>
experiment_results_explanations = """The project successfully built a predictive model to identify potential draft picks. While an initial Logistic Regression model struggled, the final XGBoost model combined with SMOTE effectively handled the data imbalance and achieved high performance in identifying drafted players. This demonstrates that machine learning can be a valuable tool for improving talent scouting efficiency and accuracy."""

In [ ]:
# Do not modify this code
print_tile(size="h2", key='experiment_results_explanations', value=experiment_results_explanations)